# Advanced Gated Recurrent Unit (GRU) Sentiment Analysis

This notebook builds an advanced GRU model for binary sentiment classification using the IMDB movie review dataset.

**Goal:** predict whether a movie review is positive or negative.

## 1. Why GRU?

A simple RNN updates one hidden state at every time step, but it often struggles with long sequences because important information can fade away.

A GRU improves this using gates:

- **Update gate:** decides how much old memory to keep.
- **Reset gate:** decides how much past context to ignore.
- **Candidate state:** proposes new memory.

This makes GRU useful for text, speech, time series, and other sequence problems.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, GRU, Bidirectional, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print('TensorFlow version:', tf.__version__)

## 2. Load IMDB Dataset

The dataset contains movie reviews encoded as integer word IDs.

- `1` means positive sentiment.
- `0` means negative sentiment.

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 250

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=VOCAB_SIZE)

print('Training samples:', len(x_train))
print('Testing samples:', len(x_test))
print('Example encoded review length:', len(x_train[0]))
print('Example label:', y_train[0])

## 3. Decode One Review

Keras keeps a word index for the IMDB dataset. We can decode integer sequences back into readable text.

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = {value + 3: key for key, value in word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'
reverse_word_index[3] = '<UNUSED>'

def decode_review(encoded_review):
    return ' '.join(reverse_word_index.get(token, '<UNK>') for token in encoded_review)

print(decode_review(x_train[0])[:1000])
print('\nSentiment:', 'Positive' if y_train[0] == 1 else 'Negative')

## 4. Pad Sequences

Neural networks process batches more efficiently when sequences have the same length. Reviews longer than `MAX_LEN` are truncated, and shorter reviews are padded.

In [ ]:
x_train_pad = pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test_pad = pad_sequences(x_test, maxlen=MAX_LEN, padding='post', truncating='post')

print('Padded training shape:', x_train_pad.shape)
print('Padded testing shape:', x_test_pad.shape)

## 5. Build Advanced GRU Model

This model uses:

- Embedding layer to convert word IDs into dense vectors
- Bidirectional GRU to read sequence context forward and backward
- Dropout to reduce overfitting
- Batch normalization for training stability
- Sigmoid output for binary classification

In [ ]:
EMBEDDING_DIM = 128
GRU_UNITS = 128

model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    Bidirectional(GRU(GRU_UNITS, return_sequences=True, dropout=0.25, recurrent_dropout=0.15)),
    Bidirectional(GRU(64, dropout=0.25, recurrent_dropout=0.15)),
    BatchNormalization(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Train The Model

The callbacks below make training more professional:

- `EarlyStopping` stops training when validation loss stops improving.
- `ReduceLROnPlateau` lowers the learning rate when the model gets stuck.
- `ModelCheckpoint` saves the best model.

In [ ]:
os.makedirs('models', exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=1, min_lr=1e-5),
    ModelCheckpoint('models/best_gru_sentiment.keras', monitor='val_accuracy', save_best_only=True)
]

history = model.fit(
    x_train_pad,
    y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

## 7. Plot Training Curves

In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(history.history['loss'], label='Training loss')
    axes[0].plot(history.history['val_loss'], label='Validation loss')
    axes[0].set_title('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()

    axes[1].plot(history.history['accuracy'], label='Training accuracy')
    axes[1].plot(history.history['val_accuracy'], label='Validation accuracy')
    axes[1].set_title('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()

    plt.show()

plot_history(history)

## 8. Evaluate On Test Data

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test_pad, y_test, verbose=0)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

probabilities = model.predict(x_test_pad, batch_size=128).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(classification_report(y_test, predictions, target_names=['Negative', 'Positive']))

In [ ]:
cm = confusion_matrix(y_test, predictions)
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])
display.plot(cmap='Blues')
plt.title('GRU Sentiment Confusion Matrix')
plt.show()

## 9. Predict Custom Reviews

The IMDB dataset uses its own word index. This helper encodes normal text into the same integer format.

In [ ]:
def encode_custom_review(text):
    tokens = text.lower().replace('.', ' ').replace(',', ' ').replace('!', ' ').replace('?', ' ').split()
    encoded = [1]
    for token in tokens:
        encoded.append(word_index.get(token, 2) + 3)
    encoded = [token if token < VOCAB_SIZE else 2 for token in encoded]
    return pad_sequences([encoded], maxlen=MAX_LEN, padding='post', truncating='post')

def predict_sentiment(text):
    encoded = encode_custom_review(text)
    probability = float(model.predict(encoded, verbose=0)[0][0])
    label = 'Positive' if probability >= 0.5 else 'Negative'
    confidence = probability if probability >= 0.5 else 1 - probability
    return label, confidence, probability

custom_reviews = [
    'The movie was beautiful, emotional, and wonderfully acted.',
    'This was boring, slow, and painfully predictable.',
    'The story had some weak parts, but the ending was powerful.'
]

for review in custom_reviews:
    label, confidence, probability = predict_sentiment(review)
    print(review)
    print(f'Prediction: {label} | confidence={confidence:.2%} | positive_probability={probability:.4f}\n')

## 10. Simple RNN vs GRU Discussion

A Simple RNN has one main recurrent transformation. It can work for short sequences, but it often forgets older information.

A GRU uses gates to control memory flow. This helps it keep useful context from earlier words while ignoring less useful information.

For sentiment analysis, this matters because words like **not**, **but**, **however**, and **excellent** can change the meaning of an entire review.

## 11. Conclusion

This notebook implemented an advanced GRU sentiment classifier using the IMDB dataset. It includes sequence preprocessing, a bidirectional stacked GRU model, training callbacks, evaluation metrics, plots, and custom predictions.

Possible next steps:

- compare GRU with LSTM
- add attention
- use pretrained word embeddings
- train on a Kaggle review dataset
- deploy the model with Streamlit or Flask